# 09 – Stance labelling, stance baseline and actors (NER)

1. **Stance labelling set:** 60 texts (30 chatbot answers, 30 news chunks), a codebook with an example for each label, sheets for two labellers (20 texts both label, 20 only A, 20 only B), and a kappa script tested on dummy labels.
2. **Stance baseline:** VADER-based and TF-IDF + logistic regression classifiers scored against the human labels with cross-validation. The code is written and tested on dummy data; it runs on real labels once the team has produced them.
3. **Actors:** spaCy named-entity recognition over both corpora, actor names normalised (for example `EU Commission` and `European Commission` become one actor), top-actor tables per source and region.

**Labelling is left to the team.** No stance labels are produced here; the dummy labels used in the tests are never saved.

In [1]:
import json, re, tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from agreement import agreement_report, cohen_kappa

CFG = json.load(open("config.json"))
OUT = Path(CFG["analysis_dir"]); FR = Path("dataset/frames"); DATA = Path("data")
SEED = 42
STANCE = ["for", "against", "mixed", "none"]

## 1. Stance labelling set
**Question the labellers answer:** what is the text's position on *government regulation of AI* (or on the specific rule it discusses)?
`for` (favours regulation or stricter rules) · `against` (opposes them) · `mixed` (weighs both sides, or reports both without favouring one) · `none` (takes no position, or is not about regulation, for example a product launch). The plan names for / against / mixed; `none` is added because many news chunks take no position and forcing a label on them would add noise. Kappa is reported on all four labels and on the three plan labels.

**Design:** stratified sample (chatbot: 5 answers for each model × wording; news: 8 US, 7 UK, 7 EU, 8 Other), seeded. 20 shared texts (10 chatbot, 10 news) go to both labellers; 20 only to A; 20 only to B. Opaque ids, shuffled differently per labeller, no source column, no frame scores. With only 20 shared texts the kappa has a wide interval: say so in the report.

In [2]:
def head_words(text, n=300):
    for k, m in enumerate(re.finditer(r"\S+", text)):
        if k == n - 1:
            return text[:m.end()]
    return text

ans = pd.read_json(CFG["answers_jsonl"], lines=True); ans = ans[ans["chatbot"].isin(["llama3.1:8b", "qwen3.5:9b"])].reset_index(drop=True)
news = pd.read_csv(CFG["news_corpus_csv"], keep_default_na=False)

rng = np.random.default_rng(SEED)
ch = pd.concat([g.sample(5, random_state=SEED) for _, g in ans.groupby(["chatbot", "wording"])])
nw = pd.concat([news[news["region"] == r].sample(n, random_state=SEED) for r, n in {"US": 8, "UK": 7, "EU": 7, "Other": 8}.items()])
ch_items = [{"source": "chatbot", "ref": f"{r.prompt_id}|r{r.replicate}|{r.chatbot}", "text": head_words(r.answer)} for r in ch.itertuples()]
nw_items = [{"source": "news", "ref": r.doc_id, "text": head_words(r.text)} for r in nw.itertuples()]
rng.shuffle(ch_items); rng.shuffle(nw_items)
items = pd.DataFrame(ch_items + nw_items)
items["set"] = (["shared"] * 10 + ["A"] * 10 + ["B"] * 10) * 2          # chatbot rows first, then news rows: 10 shared, 10 A, 10 B in each
ids = rng.permutation(len(items)) + 1
items["stance_id"] = [f"S{i:03d}" for i in ids]
key = items[["stance_id", "source", "ref", "set"]].sort_values("stance_id")
INSTR = ("INSTRUCTIONS (row 0, ignore when scoring): put in the stance column what the text's position is on government regulation of AI: "
         "for, against, mixed or none. Examples and rules: dataset/frames/stance_codebook.md. Label alone; do not discuss with the other labeller until both are finished.")
def sheet_for(who, seed):
    s = items[items["set"].isin(["shared", who])].sample(frac=1, random_state=seed)
    return pd.concat([pd.DataFrame([{"stance_id": "S000", "text": INSTR, "stance": "", "note": ""}]), s[["stance_id", "text"]].assign(stance="", note="")], ignore_index=True)
sheet_for("A", 1).to_csv(DATA / "stance_sheet_A.csv", index=False); sheet_for("B", 2).to_csv(DATA / "stance_sheet_B.csv", index=False)
key.to_csv(DATA / "stance_key.csv", index=False)
print(len(items), "texts;", items["set"].value_counts().to_dict(), "| each sheet has", 40, "texts + 1 instruction row | sources in the shared set:", items[items.set == "shared"]["source"].value_counts().to_dict())

60 texts; {'shared': 20, 'A': 20, 'B': 20} | each sheet has 40 texts + 1 instruction row | sources in the shared set: {'chatbot': 10, 'news': 10}


In [3]:
codebook = '''# Stance codebook

**Unit:** one text (the first 300 words of an answer or article).
**Question:** what is the text's position on **government regulation of AI** (or on the specific rule or measure it discusses)?

| label | use it when | invented example (not from the data) |
|---|---|---|
| for | the text favours rules, oversight or restrictions, or argues they are needed | "Strong rules are needed because unchecked AI systems can harm people, and companies will not police themselves." |
| against | the text opposes rules or argues they would do more harm than good | "Heavy-handed AI rules would stifle innovation and push companies and talent to countries with lighter regulation." |
| mixed | the text weighs both sides, or reports both sides without favouring one, or supports some rules and opposes others | "Rules could protect people from harm, but if they are too strict they risk slowing down small companies; the details matter." |
| none | no position on regulation, or the text is not about regulation | "The company unveiled a new AI assistant with a faster model and a longer battery life." |

**Rules**
1. Label the text's own position, not the topic. A text about a ban is not "for" the ban unless it endorses it.
2. A quoted opinion counts only if the text clearly adopts it. A report that quotes one supporter and one critic is `mixed`.
3. Descriptive reporting of a new law with no evaluation is `none`.
4. If a text argues one side but admits a drawback in a sentence, label the side it argues (`for` or `against`), not `mixed`.
5. Chatbot answers to "why is it a good idea" or "why is it a bad idea" prompts often argue one side: label what the text says, not what the prompt asked.
6. If unsure between a side and `mixed`, choose `mixed`; if unsure between `mixed` and `none`, choose `none` only when regulation is not discussed.
7. Label alone. Compare after both labellers have finished.

**Agreement:** the 20 shared texts give Cohen's kappa (all four labels, and the three labels for / against / mixed). Disputed shared texts are discussed and given one consensus label.
'''
(FR / "stance_codebook.md").write_text(codebook); print("written: dataset/frames/stance_codebook.md")

written: dataset/frames/stance_codebook.md


### The kappa script (tested on dummy labels)
`score_stance` reads the two labelled sheets, takes the texts both labelled, and reports observed agreement, Cohen's kappa on four labels and on the three plan labels, the confusion matrix and the disputed texts. `build_consensus` produces one label per text (agreed label for shared texts that match; `DISPUTED` for shared texts to discuss; the single labeller's label for A-only and B-only texts).

In [4]:
def read_sheet(path):
    # returns (labels without skipped texts, set of skipped ids, set of all ids on the sheet); 'skip' = the labeller skipped a distressing text
    s = pd.read_csv(path, keep_default_na=False); s = s[s["stance_id"] != "S000"]
    all_ids = set(s["stance_id"]); skipped = set(s.loc[s["stance"] == "skip", "stance_id"]); s = s[s["stance"] != "skip"]
    bad = sorted(set(s["stance"]) - set(STANCE))
    if bad or (s["stance"] == "").any():
        raise ValueError(f"{path}: empty or invalid labels {bad}")
    return s[["stance_id", "stance"]].set_index("stance_id")["stance"], skipped, all_ids

def score_stance(path_A, path_B, key_path=DATA / "stance_key.csv", save=True):
    (A, skA, idsA), (B, skB, idsB) = read_sheet(path_A), read_sheet(path_B)
    expected = set(pd.read_csv(key_path)[lambda d: d["set"] == "shared"]["stance_id"])
    if idsA & idsB != expected:
        raise ValueError(f"the two sheets share {len(idsA & idsB)} texts, expected {len(expected)} shared texts from the key")
    shared = sorted(set(A.index) & set(B.index))             # shared texts that both labellers labelled
    a, b = A.loc[shared].to_numpy(), B.loc[shared].to_numpy()
    rep4 = agreement_report(a, b, STANCE)
    three = np.isin(a, STANCE[:3]) & np.isin(b, STANCE[:3])
    k3 = cohen_kappa(a[three], b[three], STANCE[:3]) if three.sum() > 1 else float("nan")
    res = {"n_shared": len(shared), "skipped_shared": len(expected) - len(shared), "agreement": rep4["accuracy"], "kappa_4_labels": rep4["kappa"], "n_three_label": int(three.sum()), "kappa_3_labels": k3,
           "confusion": rep4["confusion"], "disputed": [s for s, x, y in zip(shared, a, b) if x != y]}
    if save:
        pd.DataFrame([{k: v for k, v in res.items() if k not in ("confusion", "disputed")}]).round(3).to_csv(OUT / "stance_agreement.csv", index=False)
        res["confusion"].to_csv(OUT / "stance_agreement_confusion.csv")
    return res

def build_consensus(path_A, path_B):
    (A, skA, _), (B, skB, _) = read_sheet(path_A), read_sheet(path_B); rows = {}
    for s in A.index.union(B.index):
        if s in A.index and s in B.index: rows[s] = A[s] if A[s] == B[s] else "DISPUTED"
        else: rows[s] = A[s] if s in A.index else B[s]
    for s in skA & skB: rows[s] = "SKIPPED"                 # skipped by everyone who was asked: no label
    return pd.Series(rows, name="stance")

def landis_koch(k):
    return "poor" if k < 0 else "slight" if k <= .2 else "fair" if k <= .4 else "moderate" if k <= .6 else "substantial" if k <= .8 else "almost perfect"

# ---- tests with DUMMY labels (temporary files, nothing saved in the repo) ----
rngt = np.random.default_rng(SEED); kk = pd.read_csv(DATA / "stance_key.csv"); shared_ids = kk[kk["set"] == "shared"]["stance_id"].tolist()
def dummy_sheet(who, agree_with=None, p=1.0):
    s = pd.read_csv(DATA / f"stance_sheet_{who}.csv", keep_default_na=False)
    labs = []
    for sid in s["stance_id"]:
        if sid == "S000": labs.append("")
        elif agree_with is not None and sid in agree_with and rngt.random() < p: labs.append(agree_with[sid])
        else: labs.append(rngt.choice(STANCE))
    return s.assign(stance=labs)
with tempfile.TemporaryDirectory() as td:
    td = Path(td); sA = dummy_sheet("A"); pA = td / "A.csv"; sA.to_csv(pA, index=False)
    truth = dict(zip(sA["stance_id"], sA["stance"]))
    pB = td / "B_same.csv"; dummy_sheet("B", truth, 1.0).to_csv(pB, index=False)
    r = score_stance(pA, pB, save=False); assert r["agreement"] == 1 and abs(r["kappa_4_labels"] - 1) < 1e-12 and r["disputed"] == [], r
    pB2 = td / "B_random.csv"; dummy_sheet("B").to_csv(pB2, index=False); r2 = score_stance(pA, pB2, save=False)
    assert r2["agreement"] < 0.7, r2
    # an invalid label must stop the script
    bad = dummy_sheet("B", truth, 1.0); bad.loc[3, "stance"] = "maybe"; pbad = td / "B_bad.csv"; bad.to_csv(pbad, index=False)
    try: score_stance(pA, pbad, save=False); raise AssertionError("invalid label was accepted")
    except ValueError: pass
    skip_b = dummy_sheet("B", truth, 1.0); skip_b.loc[skip_b["stance_id"] == shared_ids[0], "stance"] = "skip"; pskip = td / "B_skip.csv"; skip_b.to_csv(pskip, index=False)
    rs = score_stance(pA, pskip, save=False); assert rs["skipped_shared"] == 1 and rs["n_shared"] == 19 and rs["agreement"] == 1, rs
    cons = build_consensus(pA, pB2); assert len(cons) == 60 and (cons == "DISPUTED").sum() == len(r2["disputed"])
print("stance kappa script: identical sheets -> kappa 1; random labellers -> agreement", round(r2["agreement"], 2), "kappa", round(r2["kappa_4_labels"], 2),
      "; invalid label rejected; a skipped shared text is dropped and counted; consensus keeps", int((cons == 'DISPUTED').sum()), "disputed texts for discussion")

stance kappa script: identical sheets -> kappa 1; random labellers -> agreement 0.3 kappa 0.07 ; invalid label rejected; a skipped shared text is dropped and counted; consensus keeps 14 disputed texts for discussion


**To score real labels:** save the two labelled sheets (for example `data/stance_sheet_A_labelled.csv` and `data/stance_sheet_B_labelled.csv`), then run
`res = score_stance("data/stance_sheet_A_labelled.csv", "data/stance_sheet_B_labelled.csv")`, read `res["kappa_4_labels"]`, `res["disputed"]` (texts to discuss), and `landis_koch(kappa)` for the verbal label. After the discussion, write the agreed label for each disputed text into a copy of `build_consensus(...)` and save it as `data/stance_labels_consensus.csv` (columns `stance_id`, `stance`) for the baseline below.

## 2. Stance baseline (written and tested; runs when consensus labels exist)
Two baselines against the human stance labels, stratified cross-validation, with a majority-class baseline for reference:
- **VADER:** sentence-level compound scores summarised as four features (mean, standard deviation, share of positive and of negative sentences) fed to a logistic regression. VADER measures tone, not stance, so it is expected to be weak: that is the point of the baseline.
- **TF-IDF + logistic regression:** unigrams and bigrams.
The number of folds is the largest of 2–5 that every class can fill; classes with fewer than 2 texts are reported and left out of the cross-validation.

In [5]:
sia = SentimentIntensityAnalyzer()
def vader_features(text):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    c = np.array([sia.polarity_scores(s)["compound"] for s in sents]) if sents else np.zeros(1)
    return [c.mean(), c.std(), (c > 0.05).mean(), (c < -0.05).mean()]

def stance_baseline(df, seed=SEED):
    """df: columns text, stance. Returns a table of cross-validated accuracy, macro-F1 and kappa vs the human labels."""
    counts = df["stance"].value_counts(); usable = counts[counts >= 2].index
    d = df[df["stance"].isin(usable)].reset_index(drop=True); k = int(min(5, counts[usable].min())); notes = {"left_out_classes": sorted(set(counts.index) - set(usable)), "folds": k}
    if k < 2 or d["stance"].nunique() < 2: raise ValueError("not enough labelled texts per class for cross-validation")
    X_v = np.array([vader_features(t) for t in d["text"]]); y = d["stance"].to_numpy()
    models = {"majority class": None, "VADER features + logistic regression": ("v", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))),
              "TF-IDF (1-2 grams) + logistic regression": ("t", make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, class_weight="balanced", C=3)))}
    rows = []
    for name, spec in models.items():
        pred = np.empty(len(d), dtype=object)
        for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(d, y):
            if spec is None: pred[te] = pd.Series(y[tr]).mode()[0]; continue
            kind, m = spec; Xtr, Xte = (X_v[tr], X_v[te]) if kind == "v" else (d["text"].iloc[tr], d["text"].iloc[te])
            pred[te] = m.fit(Xtr, y[tr]).predict(Xte)
        rows.append({"model": name, "n": len(d), "accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"), "kappa_vs_human": cohen_kappa(y, pred, sorted(set(y)))})
    return pd.DataFrame(rows).round(3), notes

# ---- tests on DUMMY data: separable synthetic texts, then random labels ----
words = {"for": "support strong rules necessary protect safeguard essential", "against": "oppose burden stifle harmful restrictive unnecessary", "mixed": "however both balance trade-off depends careful",
         "none": "launch product feature release update announced device"}
r0 = np.random.default_rng(1)
synth = pd.DataFrame([{"text": " ".join(r0.choice(words[c].split(), 12)) + ". " + " ".join(r0.choice("the a of and AI new said will".split(), 8)), "stance": c} for c in STANCE for _ in range(15)])
tab, notes = stance_baseline(synth); tf = tab.set_index("model").loc["TF-IDF (1-2 grams) + logistic regression"]
assert tf["accuracy"] > 0.9, tab
shuf = synth.assign(stance=r0.permutation(synth["stance"].to_numpy())); tab2, _ = stance_baseline(shuf)
assert tab2.set_index("model").loc["TF-IDF (1-2 grams) + logistic regression", "kappa_vs_human"] < 0.3, tab2
print("stance baseline tests passed: separable dummy data -> TF-IDF accuracy", tf["accuracy"], "; shuffled labels -> kappa", tab2.set_index("model").loc["TF-IDF (1-2 grams) + logistic regression", "kappa_vs_human"])
tab

stance baseline tests passed: separable dummy data -> TF-IDF accuracy 1.0 ; shuffled labels -> kappa -0.044


,model,n,accuracy,macro_f1,kappa_vs_human
0,majority class,60,0.250,0.100,0.000
1,VADER features + logistic regression,60,0.883,0.885,0.844
2,TF-IDF (1-2 grams) + logistic regression,60,1.000,1.000,1.000


In [6]:
CONSENSUS = DATA / "stance_labels_consensus.csv"
if CONSENSUS.exists():
    lab = pd.read_csv(CONSENSUS, keep_default_na=False); lab = lab[lab["stance"] != "DISPUTED"]
    sheetA, sheetB = pd.read_csv(DATA / "stance_sheet_A.csv", keep_default_na=False), pd.read_csv(DATA / "stance_sheet_B.csv", keep_default_na=False)
    texts = pd.concat([sheetA, sheetB]).drop_duplicates("stance_id").set_index("stance_id")["text"]
    df = lab.assign(text=lab["stance_id"].map(texts)).dropna(subset=["text"])
    res_tab, notes = stance_baseline(df); res_tab.to_csv(OUT / "stance_baseline.csv", index=False)
    print(notes); display(res_tab)
else:
    print("No", CONSENSUS, "yet: the baseline waits for the human labels (see the instructions above).")

No data/stance_labels_consensus.csv yet: the baseline waits for the human labels (see the instructions above).


## 3. Actors: named entities with normalised names
spaCy `en_core_web_sm` named-entity recognition on the first 300 words of every text (the same text the frame scorer sees). Actors are `ORG`, `PERSON` and `GPE` entities. Names are normalised to one form (case, a leading *the*, possessive *'s*, then an alias table: `EU Commission` = `European Commission`, `Trump` = `Donald Trump`, `U.S.` = `United States`, and so on). AI product names that the recogniser tags as organisations (`ChatGPT`, `Gemini`, `Claude`, `Grok`) are dropped because they are products, not actors. The counts are **document frequency** (how many texts mention the actor), which is not inflated by one article repeating a name.

In [7]:
ner = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "attribute_ruler"])

ALIAS = {
 "eu": "European Union", "e.u.": "European Union", "european union": "European Union", "brussels": "European Union",
 "eu commission": "European Commission", "european commission": "European Commission", "e.u. commission": "European Commission",
 "european parliament": "European Parliament", "eu parliament": "European Parliament",
 "us": "United States", "u.s.": "United States", "u.s": "United States", "usa": "United States", "united states": "United States", "america": "United States", "united states of america": "United States",
 "uk": "United Kingdom", "u.k.": "United Kingdom", "united kingdom": "United Kingdom", "britain": "United Kingdom", "great britain": "United Kingdom",
 "openai": "OpenAI", "open ai": "OpenAI", "google": "Google", "alphabet": "Google", "google deepmind": "Google", "deepmind": "Google",
 "meta": "Meta", "meta platforms": "Meta", "facebook": "Meta", "microsoft": "Microsoft", "anthropic": "Anthropic", "xai": "xAI", "x.ai": "xAI",
 "nvidia": "Nvidia", "apple": "Apple", "amazon": "Amazon", "tesla": "Tesla", "spacex": "SpaceX",
 "white house": "White House", "congress": "US Congress", "us congress": "US Congress", "u.s. congress": "US Congress", "senate": "US Senate", "us senate": "US Senate", "u.s. senate": "US Senate",
 "pentagon": "Pentagon", "department of defense": "Pentagon", "defense department": "Pentagon", "department of war": "Pentagon",
 "ftc": "Federal Trade Commission", "federal trade commission": "Federal Trade Commission", "un": "United Nations", "united nations": "United Nations",
 "trump": "Donald Trump", "donald trump": "Donald Trump", "musk": "Elon Musk", "elon musk": "Elon Musk", "altman": "Sam Altman", "sam altman": "Sam Altman",
 "biden": "Joe Biden", "joe biden": "Joe Biden", "amodei": "Dario Amodei", "dario amodei": "Dario Amodei", "zuckerberg": "Mark Zuckerberg", "mark zuckerberg": "Mark Zuckerberg",
 "huang": "Jensen Huang", "jensen huang": "Jensen Huang", "starmer": "Keir Starmer", "keir starmer": "Keir Starmer", "sunak": "Rishi Sunak", "rishi sunak": "Rishi Sunak",
 "macron": "Emmanuel Macron", "emmanuel macron": "Emmanuel Macron", "china": "China", "beijing": "China",
 # product names that carry the company name belong to the company
 "google gemini": "Google", "google ai": "Google", "microsoft copilot": "Microsoft", "microsoft 365": "Microsoft", "meta ai": "Meta", "amazon web services": "Amazon", "aws": "Amazon",
 "openai’s chatgpt": "OpenAI", "associated press": "Associated Press", "ap": "Associated Press", "the associated press": "Associated Press",
}
PRODUCTS = {"ai", "gpt", "chatgpt", "gemini", "claude", "grok", "copilot", "llama", "gen ai", "genai", "sora", "openclaw", "siri", "a.i.", "llm", "llms"}
STATE = {"European Union", "European Commission", "European Parliament", "United States", "United Kingdom", "China", "White House", "US Congress", "US Senate", "Pentagon", "Federal Trade Commission", "United Nations"}
MEDIA = {"Reuters", "Associated Press", "Bloomberg", "CNN", "BBC", "Fox News", "The New York Times", "New York Times", "Washington Post", "Financial Times", "The Guardian", "Wall Street Journal", "Business Insider", "NBC News", "CNBC", "The Verge", "TechCrunch", "AFP"}
COMPANY = {"OpenAI", "Google", "Meta", "Microsoft", "Anthropic", "xAI", "Nvidia", "Apple", "Amazon", "Tesla", "SpaceX", "DeepSeek", "Samsung", "IBM", "Oracle", "Snowflake", "Salesforce", "Perplexity"}

def normalise_actor(text, label=None):
    t = re.sub(r"[’']s$", "", text.strip()); t = re.sub(r"^(the|The)\s+", "", t).strip(" .,;:\"“”()")
    key = re.sub(r"\s+", " ", t.lower())
    if not key or key in PRODUCTS or re.fullmatch(r"[\d\W]+", key) or len(key) < 2: return None
    return ALIAS.get(key, re.sub(r"\s+", " ", t))

def category(actor, label):
    return "state / regulator" if actor in STATE else "company" if actor in COMPANY else "news media" if actor in MEDIA else {"PERSON": "person", "GPE": "country / place"}.get(label, "other organisation")

# tests
assert normalise_actor("EU Commission") == normalise_actor("European Commission") == normalise_actor("the European Commission") == "European Commission"
assert normalise_actor("U.S.") == normalise_actor("US") == normalise_actor("the United States") == "United States"
assert normalise_actor("Trump") == normalise_actor("Donald Trump") == normalise_actor("Trump’s") == "Donald Trump"
assert normalise_actor("Google Gemini") == normalise_actor("Google") == "Google" and normalise_actor("Microsoft Copilot") == "Microsoft"
assert normalise_actor("ChatGPT") is None and normalise_actor("AI") is None and normalise_actor("2026") is None and normalise_actor("OpenAI") == "OpenAI"
print("actor normalisation tests passed")

actor normalisation tests passed


In [8]:
KEEP = {"ORG", "PERSON", "GPE"}
def actors(texts):
    out = []
    for doc in ner.pipe(texts, batch_size=32):
        ents = []
        for e in doc.ents:
            if e.label_ in KEEP:
                a = normalise_actor(e.text, e.label_)
                if a: ents.append((a, e.label_))
        out.append(ents)
    return out

corp = pd.concat([ans.assign(group=ans["chatbot"].map({"llama3.1:8b": "chatbot: Llama", "qwen3.5:9b": "chatbot: Qwen"}), text_cut=ans["answer"].map(lambda t: " ".join(t.split()[:300]))),
                  news.assign(group="news: " + news["region"], text_cut=news["text"].map(lambda t: " ".join(t.split()[:300])))], ignore_index=True)
corp["ents"] = actors(corp["text_cut"].tolist())
rows = []
for g, d in list(corp.groupby("group")) + [("news: all", corp[corp["group"].str.startswith("news")]), ("chatbot: both", corp[corp["group"].str.startswith("chatbot")])]:
    docs, ments, lab = Counter(), Counter(), {}
    for ents in d["ents"]:
        for a in {x for x, _ in ents}: docs[a] += 1
        for a, l in ents: ments[a] += 1; lab.setdefault(a, Counter())[l] += 1
    for rank, (a, n) in enumerate(docs.most_common(20), 1):
        rows.append({"source": g, "rank": rank, "actor": a, "category": category(a, lab[a].most_common(1)[0][0]), "texts_mentioning": n,
                     "share_of_texts_%": round(100 * n / len(d), 1), "mentions": ments[a], "n_texts_in_source": len(d)})
top = pd.DataFrame(rows); top.to_csv(OUT / "actors_top_by_source.csv", index=False)
top[top["source"].isin(["news: all", "chatbot: both"])].pivot_table(index="rank", columns="source", values="actor", aggfunc="first")

source,chatbot: both,news: all
rank,,
1,Google,United States
2,China,Google
3,European Union,OpenAI
4,Amazon,Microsoft
5,Microsoft,United Kingdom
6,Stricter AI,Anthropic
7,United States,Donald Trump
8,Uighur Muslims,Elon Musk
9,Alexa,California


In [9]:
# who is talked about: share of actor mentions by category, and entities per text
def cat_share(d):
    c = Counter(category(a, l) for ents in d["ents"] for a, l in ents); t = sum(c.values()) or 1
    return {**{k: round(100 * v / t, 1) for k, v in c.items()}, "entities_per_text": round(sum(len(e) for e in d["ents"]) / len(d), 1), "texts": len(d)}
shares = pd.DataFrame({g: cat_share(d) for g, d in corp.groupby("group")}).T.fillna(0)
shares.to_csv(OUT / "actor_category_shares.csv"); shares

,state / regulator,person,company,other organisation,entities_per_text,texts,country / place,news media
chatbot: Llama,26.7,26.7,33.3,13.3,0.1,108.0,0.0,0.0
chatbot: Qwen,25.0,0.0,50.0,25.0,0.0,108.0,0.0,0.0
news: EU,9.8,25.8,11.7,37.0,13.5,82.0,14.5,1.3
news: Other,2.3,23.3,13.2,48.4,13.5,104.0,11.5,1.3
news: UK,10.9,27.8,8.0,41.3,12.7,101.0,10.4,1.6
news: US,11.7,26.1,13.5,36.1,14.1,133.0,10.8,1.8


In [10]:
# which spellings were merged? (raw strings per normalised actor, for the actors with the most variants)
var = defaultdict(Counter)
for doc in ner.pipe(corp["text_cut"].tolist()[:], batch_size=32):
    for e in doc.ents:
        if e.label_ in KEEP:
            a = normalise_actor(e.text, e.label_)
            if a: var[a][e.text.strip()] += 1
merged = pd.DataFrame([{"actor": a, "raw_spellings": ", ".join(f"{s} ({n})" for s, n in c.most_common(6)), "n_spellings": len(c)} for a, c in var.items() if len(c) > 1]).sort_values("n_spellings", ascending=False)
merged.head(12)

,actor,raw_spellings,n_spellings
1,European Union,"EU (25), the European Union's (3), the Europea...",8
0,United States,"US (86), U.S. (65), the United States (22), Am...",6
8,Pentagon,"Pentagon (54), the Department of War (7), the ...",6
10,Meta,"Meta (36), Meta AI (13), Meta Platforms (3), M...",5
3,Google,"Google (98), Google Gemini (23), Google DeepMi...",5
7,Donald Trump,"Trump (56), Donald Trump (28), Trump’s (3), Tr...",5
26,Associated Press,"AP (10), The Associated Press (4), Associated ...",4
24,White House,"White House (6), The White House (4), the Whit...",4
17,United Kingdom,"UK (98), Britain (16), the United Kingdom (5),...",4
12,European Commission,"The European Commission (4), the European Comm...",4


## Limits to state in the report
- The recogniser is a small general-English model (`en_core_web_sm`); it mislabels some entities and misses others. The alias table covers the most frequent actors only; spellings it does not know stay separate.
- Actor tables use the first 300 words of each text, the same text the frame scorer sees, so actors named later in an article are not counted.
- Chatbot answers name almost no entities (about 0.1 per answer for Llama, 0.0 for Qwen) because the prompts are generic policy questions; the few that appear include recogniser errors (for example `Stricter AI`, `Governmental`, `Healthcare`), so the chatbot-vs-news actor comparison says little beyond that chatbots do not name actors.
- Stance labelling has only 20 shared texts, so kappa has a wide interval; two labellers from the same team may share assumptions.